In [1]:

# Import necessary libraries
import os, mne, re
from mne.channels import make_standard_montage
from glob import glob
from mne_bids import BIDSPath, write_raw_bids, read_raw_bids
import numpy as np
from mne_icalabel import label_components                
from pyprep.find_noisy_channels import NoisyChannels

root_gen = 'C:/Users/mfbpe/Desktop/DATA/2025_Valuation/'

# Set the folder for recreate the data in bids format downsampled 
root_bids = root_gen + 'bids'

# Set the derivatives directory path
root_derivatives = root_gen + 'derivatives'

# Set the image directory path
root_image = root_gen + 'images_prep_eeg'

# Change the current working directory to the derivatives directory
os.chdir(root_derivatives)

# Get a list of all file paths with .bdf extension in the source directory
all_files_path = sorted(glob(f'{root_bids}/*/*/*_eeg.vhdr*'), key=len)

prtp_chs_out = {'1': ['FC5', 'P1'], '2': ['P2', 'PO4', 'AF8'], '3': ['T8', 'P2', 'POz'], '4': ['F3', 'Iz', 'POz', 'Oz', 'PO3'], '5': ['P8', 'T7', 'O1', 'POz', 'TP8', 'PO3', 'FT7', 'Pz', 'TP7'], '6': ['Pz', 'T7', 'POz', 'PO3'], '7': ['O1', 'CP3', 'Iz', 'POz', 'P1', 'PO3', 'Pz', 'P3'], '8': ['P1', 'PO3'], '9': ['POz'], '10': ['FT7', 'T7', 'P1', 'T8'], '11': ['PO7', 'P10', 'POz'], '12': ['T7', 'F7', 'FT8'], '13': ['O1', 'Iz', 'POz', 'PO3'], '14': ['Oz', 'O1', 'Iz', 'P9'], '15': ['PO4', 'P1', 'POz', 'PO3'], '16': ['CP5', 'P2', 'PO4', 'Iz', 'P1', 'PO3', 'Pz', 'P3'], '17': ['PO4', 'P10', 'POz'], '18': ['O1', 'F6', 'P5', 'P2', 'F3', 'POz', 'AF8', 'PO3', 'P3'], '19': ['P2', 'POz', 'Fp1', 'AF7', 'P1', 'Fp2'], '20': ['Iz'], '21': ['FT8', 'P5', 'POz', 'AF7', 'F4'], '22': ['T8', 'P2', 'CP4'], '23': ['O2', 'P1', 'PO3'], '24': ['T8', 'PO7', 'P5', 'P4', 'POz', 'P7', 'CP4', 'PO3', 'C4'], '25': ['PO7', 'P5', 'POz', 'PO3'], '26': ['T7', 'P1', 'P3'], '27': ['PO3'], '28': ['P8', 'PO4', 'POz', 'P1', 'C6'], '29': ['P4'], '30': ['POz', 'Pz', 'F5', 'P9', 'C4'], '31': ['F8', 'P10'], '32': ['P2', 'POz', 'P1', 'Pz', 'P3', 'TP7'], '33': ['P2', 'PO4', 'POz', 'PO3', 'Pz', 'TP7'], '34': ['POz', 'P1', 'Fp2', 'CP4', 'Pz', 'TP7'], '35': ['PO7', 'T7', 'PO4', 'PO3', 'FT7', 'TP7'], '36': ['PO7', 'P2', 'POz', 'P7', 'PO3', 'Pz', 'P9', 'TP7'], '37': ['PO7', 'CP3', 'P4', 'POz', 'P7', 'AF7', 'P1', 'TP7', 'C3'], '38': ['P5', 'POz', 'Oz', 'P1', 'PO3', 'Pz', 'P3'], '39': ['T8', 'T7', 'P2', 'P1', 'PO3', 'P3'], '40': ['Oz', 'Pz', 'P3']}
 # dictionary to store the bad channels for each participant, to avoid re-detecting them if they are already detected in a previous participant
prtp_ica_out = {'1': [[0, 1], np.int64(63)], '2': [[0, 1, 2], np.int64(63)], '3': [[1, 4], np.int64(62)], '4': [[0, 3], np.int64(63)], '5': [[0, 1, 3], np.int64(63)], '6': [[0, 1], np.int64(63)], '7': [[1, 2], np.int64(63)], '8': [[0, 1], np.int64(63)], '9': [[0, 1], np.int64(63)], '10': [[0, 1], np.int64(63)], '11': [[1, 2, 3], np.int64(63)], '12': [[0], np.int64(63)], '13': [[0, 1, 20], np.int64(63)], '14': [[1, 3], np.int64(63)], '15': [[0, 1], np.int64(63)], '16': [[0, 1], np.int64(63)], '17': [[0, 1], np.int64(63)], '18': [[0, 3], np.int64(63)], '19': [[0, 2], np.int64(63)], '20': [[0, 1, 37], np.int64(63)], '21': [[3, 4], np.int64(38)], '22': [[0, 1], np.int64(63)], '23': [[0, 2], np.int64(63)], '24': [[0, 1], np.int64(63)], '25': [[1, 3], np.int64(63)], '26': [[0, 1, 2], np.int64(63)], '27': [[0, 1, 2], np.int64(63)], '28': [[0, 24], np.int64(63)], '29': [[1, 2], np.int64(61)], '30': [[3, 10], np.int64(53)], '31': [[17], np.int64(63)], '32': [[4], np.int64(20)], '33': [[0, 1], np.int64(63)], '34': [[0, 2], np.int64(63)], '35': [[1, 4], np.int64(48)], '36': [[0, 1], np.int64(63)], '37': [[2], np.int64(19)], '38': [[1, 2], np.int64(61)], '39': [[0, 1, 2], np.int64(63)], '40': [[0, 1, 7], np.int64(63)]}
 # dictionary to store the ICA components removed for each participant, to avoid re-detecting them if they are already detected in a previous participant



In [4]:
%%capture 
# Extract the participant number from the file path
n_part = '32'
bidspath = BIDSPath(subject=n_part, task='valuation', datatype='eeg', root=root_bids)
raw = read_raw_bids(bids_path=bidspath).load_data()


In [5]:
%matplotlib qt
raw.plot(duration=200, scalings=dict(eeg=200e-6))

In [ ]:

for part in all_files_path:
    # Extract the participant number from the file path
    n_part = part.split("\\")[1].split("-")[1]

    bidspath = BIDSPath(subject=n_part, task='valuation', datatype='eeg', root=root_bids)
    raw = read_raw_bids(bids_path=bidspath).load_data()

    # Apply a bandpass filter to the raw data 
    raw.filter(l_freq=.01, h_freq=40, n_jobs=-1)

    raw_copy = raw.copy().filter(1, None, n_jobs=-1) #to improve detection channels and ICA labeling IClabels

    raw.set_channel_types({'EXG1': 'eog', 'EXG2': 'eog', 'EXG3': 'eog', 'EXG4': 'eog', 'EXG5': 'emg', 'EXG6': 'emg', 'EXG7': 'emg', 'EXG8': 'emg'})
    raw_copy.set_channel_types({'EXG1': 'eog', 'EXG2': 'eog', 'EXG3': 'eog', 'EXG4': 'eog', 'EXG5': 'emg', 'EXG6': 'emg', 'EXG7': 'emg', 'EXG8': 'emg'})

    # Set the 32 system BioSemi channel positions on the data
    montage = make_standard_montage('biosemi64')
    raw.set_montage(montage, on_missing="ignore")
    
    # Interpolate bad channels for participants in the prtp_chs_out list
    if n_part in list(prtp_chs_out):
        raw.info['bads'] = prtp_chs_out[n_part]
        raw_copy.info['bads'] = prtp_chs_out[n_part]
        raw = raw.interpolate_bads()
        raw_copy = raw_copy.interpolate_bads()        
    else :
        noisy= NoisyChannels(raw_copy)
        noisy.find_all_bads()
        bad_chs = noisy.get_bads()
        raw.info['bads'] = bad_chs
        raw = raw.interpolate_bads()
        raw_copy = raw_copy.interpolate_bads()
        prtp_chs_out[n_part]=bad_chs
    
    raw.set_channel_types({'EXG1': 'eog', 'EXG2': 'eog', 'EXG3': 'eog', 'EXG4': 'eog', 'EXG5': 'emg', 'EXG6': 'emg', 'EXG7': 'emg', 'EXG8': 'emg'})
    raw_copy.set_channel_types({'EXG1': 'eog', 'EXG2': 'eog', 'EXG3': 'eog', 'EXG4': 'eog', 'EXG5': 'emg', 'EXG6': 'emg', 'EXG7': 'emg', 'EXG8': 'emg'})

    raw_copy.set_eeg_reference('average')
    raw.set_eeg_reference('average')

    try : 
        ica = mne.preprocessing.read_ica(f"sub-{n_part}_ica.fif")
    except :
        ica = mne.preprocessing.ICA(method='picard',fit_params=dict(ortho=False, extended=True),random_state=21, max_iter='auto') # dict(ortho=False, extended=True) for extended Infomax like solution with picard
        ica.fit(raw_copy)

    ic_labels = label_components(raw_copy, ica, method="iclabel")

    indexes = [i for i, (l, p) in enumerate(zip(ic_labels['labels'], ic_labels['y_pred_proba']))
            if l in ['eye blink'] and p > 0.7] # categorize labels eyes artifacts with a threshold of 0.7

    ica.exclude = indexes
    ica.apply(raw)
    prtp_ica_out[n_part]=[indexes, ica.n_components_]

    fig = ica.plot_components(picks=np.arange(ica.n_components_), show = False) #create figures of the components with the one removed
    fig.savefig(f"{root_image}sub-{n_part}_ICA.png")
    ica.save(f"sub-{n_part}_ica.fif", overwrite=True)
    # Save the preprocessed raw data
    raw.save(f"sub-{n_part}_raw.fif", overwrite=True)

    del ica, raw, raw_copy

# create the txt file

with open(f"list_bad_channels.txt", 'w') as f:
    f.write('dict = ' + repr(prtp_chs_out) + '\n')

with open(f"list_icas.txt", 'w') as f:
    f.write('dict = ' + repr(prtp_ica_out) + '\n')    

    